# 05 — Telugu ASR Improved Training (EXP-1)
## Wav2Vec2 XLS-R 300M · CTC · EXP-1 Hyperparameters

**Prerequisites (run in order):**
- `01_data_exploration_and_cleaning.ipynb` → `./telugu_asr_clean_dataset/`
- `02_model_pipeline_prototyping.ipynb` → `./telugu_wav2vec2_processor/`
- `04_multi_dataset_preparation.ipynb` → `./telugu_multi_dataset/`, `./telugu_wav2vec2_processor_v2/` *(optional but strongly recommended)*

**EXP-1 changes from baseline (`03_full_scale_training.ipynb`):**

| Parameter | Baseline | EXP-1 | Why |
|---|---|---|---|
| `mask_feature_prob` | 0.004 | **0.1** | **The key fix** — 0.004 is effectively off; standard wav2vec2 fine-tuning uses 0.1 |
| `LR` | 3e-4 | **1e-4** | Finer weight updates, avoids overshooting in late training |
| `warmup_steps` | 500 | **1000** | ~1 full epoch warmup stabilises gradient norms before cosine decay |
| `mask_time_prob` | 0.075 | **0.1** | Stronger time masking for Telugu consonant clusters |
| `epochs` | 30 | **50** | More budget; early stopping (patience=8) gates actual length |
| `early_stopping_patience` | 5 | **8** | Cosine tail can still yield gains over 6–7 consecutive eval windows |
| `early_stopping_threshold` | 0.001 | **0.005** | 0.001 WER is noise at 3.5k eval samples; 0.005 ≈ 18 words |

**Expected WER:** ~38–42% (EXP-1 alone, MSCIL only) · ~32–38% (with multi-dataset from notebook 04)  
**Environment variable required:** `HF_TOKEN` (no hardcoded keys in this notebook)

In [1]:
# Install / upgrade required packages
# datasets<3.0 keeps soundfile-based audio decoding (no torchaudio dependency)
!pip install -q \
    "typing_extensions>=4.6" \
    "transformers>=4.46" \
    "datasets<3.0" \
    "accelerate>=0.26" \
    "evaluate" \
    "jiwer" \
    "soundfile" \
    "librosa" \
    "pyctcdecode"

# kenlm Python bindings (needed only if USE_KENLM = True in the last cell)
!pip install -q kenlm 2>/dev/null || echo "kenlm install skipped (optional)"

print("Installation complete.")

Installation complete.


In [2]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124


In [ ]:
import logging
import os
import re
import sys
import json
import subprocess
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Union

import numpy as np
import torch
import evaluate
from datasets import load_from_disk, Audio
from transformers import (
    Wav2Vec2ForCTC,
    Wav2Vec2Processor,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from huggingface_hub import login

# Suppress numba DEBUG flood (triggered by librosa imports)
logging.getLogger("numba").setLevel(logging.WARNING)
logging.getLogger("numba.core").setLevel(logging.WARNING)
logging.getLogger("numba.core.ssa").setLevel(logging.WARNING)

# Logging: INFO to console, DEBUG to file
LOG_FILE = "training_improved.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler(LOG_FILE, mode="a", encoding="utf-8"),
    ],
)
logger = logging.getLogger("telugu_asr_improved")

# HuggingFace login — set HF_TOKEN env var before running
# e.g.  export HF_TOKEN="hf_..."
_hf_token = os.environ.get("HF_TOKEN")
if _hf_token:
    login(token=_hf_token)
    logger.info("HuggingFace login successful.")
else:
    logger.warning("HF_TOKEN not set — private dataset access may fail.")

logger.info("Imports complete.")


In [4]:
# ── Model & output paths ───────────────────────────────────────────────────
MODEL_NAME = "facebook/wav2vec2-xls-r-300m"
OUTPUT_DIR = "./wav2vec2-telugu-improved"

# ── EXP-1 training schedule ────────────────────────────────────────────────
LR         = 1e-4    # baseline was 3e-4
WARMUP     = 1000    # baseline was 500
EPOCHS     = 50      # baseline was 30; early stopping will gate
EVAL_STEPS = 500
SAVE_STEPS = 500
SPLIT_RATIO = 0.1
SEED        = 42
FP16        = True

# ── EXP-1 SpecAugment + dropout ───────────────────────────────────────────
MASK_TIME_PROB    = 0.1     # baseline was 0.075
MASK_TIME_LENGTH  = 10
MASK_FEAT_PROB    = 0.1     # baseline was 0.004 — THE KEY FIX
ATTENTION_DROPOUT = 0.1
HIDDEN_DROPOUT    = 0.1
FEAT_PROJ_DROPOUT = 0.0

# ── EXP-1 early stopping ──────────────────────────────────────────────────
EARLY_STOP_PATIENCE  = 8      # baseline was 5
EARLY_STOP_THRESHOLD = 0.005  # baseline was 0.001 (noise-level)

# ── Memory-adaptive batch sizing ──────────────────────────────────────────
# Effective batch = 32 across all GPU tiers
assert torch.cuda.is_available(), "No CUDA device found — check driver/runtime."
props   = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / 1e9

if vram_gb >= 40:
    BATCH_SIZE, GRAD_ACCUM = 8, 4    # A100 40/80 GB
elif vram_gb >= 20:
    BATCH_SIZE, GRAD_ACCUM = 4, 8    # A6000 / RTX 3090 / A5000
elif vram_gb >= 12:
    BATCH_SIZE, GRAD_ACCUM = 2, 16   # T4 / V100-16GB / RTX 3060 Ti
else:
    BATCH_SIZE, GRAD_ACCUM = 1, 32   # small / shared GPU

# TF32 speeds up Ampere+ matmuls with no accuracy loss
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True

print("GPU DIAGNOSTICS")
print(f"  Device     : {props.name}")
print(f"  VRAM       : {vram_gb:.1f} GB")
print(f"  Compute    : {props.major}.{props.minor}")
print()
print("EXP-1 HYPERPARAMETERS")
print(f"  LR                : {LR}")
print(f"  Warmup steps      : {WARMUP}")
print(f"  Epochs            : {EPOCHS}  (early-stop gates actual run)")
print(f"  MASK_FEAT_PROB    : {MASK_FEAT_PROB}   ← key fix (was 0.004)")
print(f"  MASK_TIME_PROB    : {MASK_TIME_PROB}")
print(f"  Early stop pat.   : {EARLY_STOP_PATIENCE}")
print(f"  Early stop thr.   : {EARLY_STOP_THRESHOLD}")
print()
print(f"BATCH CONFIG  (VRAM {vram_gb:.0f} GB tier)")
print(f"  per-device batch  : {BATCH_SIZE}")
print(f"  grad accum steps  : {GRAD_ACCUM}")
print(f"  effective batch   : {BATCH_SIZE * GRAD_ACCUM}")

GPU DIAGNOSTICS
  Device     : NVIDIA RTX A6000
  VRAM       : 51.0 GB
  Compute    : 8.6

EXP-1 HYPERPARAMETERS
  LR                : 0.0001
  Warmup steps      : 1000
  Epochs            : 50  (early-stop gates actual run)
  MASK_FEAT_PROB    : 0.1   ← key fix (was 0.004)
  MASK_TIME_PROB    : 0.1
  Early stop pat.   : 8
  Early stop thr.   : 0.005

BATCH CONFIG  (VRAM 51 GB tier)
  per-device batch  : 8
  grad accum steps  : 4
  effective batch   : 32


In [5]:
# ── Smart path detection ───────────────────────────────────────────────────
# Prefer multi-dataset output from notebook 04; fall back to MSCIL-only
DATASET_PATH = (
    "./telugu_multi_dataset"
    if Path("./telugu_multi_dataset").exists()
    else "./telugu_asr_clean_dataset"
)
PROCESSOR_PATH = (
    "./telugu_wav2vec2_processor_v2"
    if Path("./telugu_wav2vec2_processor_v2").exists()
    else "./telugu_wav2vec2_processor"
)
processor = Wav2Vec2Processor.from_pretrained(os.path.abspath(PROCESSOR_PATH), local_files_only=True)




print(f"Dataset path   : {DATASET_PATH}  {'(multi-dataset ✓)' if 'multi' in DATASET_PATH else '(MSCIL fallback — run notebook 04 for more data)'}")
print(f"Processor path : {PROCESSOR_PATH}  {'(v2 ✓)' if 'v2' in PROCESSOR_PATH else '(v1 fallback)'}")

# ── Load processor ─────────────────────────────────────────────────────────
# Note: pass path WITHOUT trailing slash to avoid HFValidationError
processor = Wav2Vec2Processor.from_pretrained(PROCESSOR_PATH)
print(f"\nProcessor vocab size : {processor.tokenizer.vocab_size}")
print(f"PAD token id         : {processor.tokenizer.pad_token_id}")

# ── Load dataset ───────────────────────────────────────────────────────────
dataset = load_from_disk(DATASET_PATH)
dataset = dataset.cast_column("audio", Audio(sampling_rate=16_000))
print(f"\nDataset size   : {len(dataset)} samples")
print(f"Columns        : {dataset.column_names}")

# ── Deterministic 90 / 10 split ───────────────────────────────────────────
split    = dataset.train_test_split(test_size=SPLIT_RATIO, seed=SEED)
train_ds = split["train"]
eval_ds  = split["test"]   # keep raw (with audio) for inference

print(f"\nTrain : {len(train_ds)} samples")
print(f"Eval  : {len(eval_ds)} samples")

Dataset path   : ./telugu_multi_dataset  (multi-dataset ✓)
Processor path : ./telugu_wav2vec2_processor_v2  (v2 ✓)

Processor vocab size : 69
PAD token id         : 68

Dataset size   : 91951 samples
Columns        : ['audio', 'transcription', 'audio_id']

Train : 82755 samples
Eval  : 9196 samples


In [6]:
def prepare_dataset(batch: dict) -> dict:
    """Waveform → input_values; transcription → label ids."""
    audio = batch["audio"]
    batch["input_values"] = processor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
    ).input_values[0]
    batch["input_length"] = len(batch["input_values"])
    batch["labels"] = processor.tokenizer(batch["transcription"]).input_ids
    return batch


# Remove all source columns; keep only what the model & collator need
KEEP_COLS = {"input_values", "input_length", "labels"}
COLS_TO_REMOVE = [c for c in train_ds.column_names if c not in KEEP_COLS]
print(f"Columns to remove : {COLS_TO_REMOVE}")

print("Mapping over train split …")
train_ds_prepared = train_ds.map(
    prepare_dataset,
    remove_columns=COLS_TO_REMOVE,
    num_proc=4,
    desc="Preparing train features",
)

print("Mapping over eval split …")
eval_ds_prepared = eval_ds.map(
    prepare_dataset,
    remove_columns=COLS_TO_REMOVE,
    num_proc=4,
    desc="Preparing eval features",
)

# Sort by length — minimises padding waste
# (replaces the removed group_by_length=True in transformers >= 4.46)
train_ds_prepared = train_ds_prepared.sort("input_length")

print(f"\nTrain prepared : {len(train_ds_prepared)} samples, sorted by input_length")
print(f"Eval  prepared : {len(eval_ds_prepared)} samples")
print(f"Sample labels (first 8): {train_ds_prepared[0]['labels'][:8]}")

Columns to remove : ['audio', 'transcription', 'audio_id']
Mapping over train split …


Preparing train features (num_proc=4): 100%|██████████| 82755/82755 [01:19<00:00, 1046.07 examples/s]


Mapping over eval split …


Preparing eval features (num_proc=4): 100%|██████████| 9196/9196 [00:15<00:00, 612.30 examples/s] 



Train prepared : 82755 samples, sorted by input_length
Eval  prepared : 9196 samples
Sample labels (first 8): [3, 1, 31, 58]


In [7]:
@dataclass
class DataCollatorCTCWithPadding:
    """
    Pad input_values and labels to the longest item in the batch.

    CRITICAL: uses -100 (not pad_token_id) as the label pad so CTC loss
    ignores padding positions.  Token-id 0 is the real Telugu character 'ం';
    using it as pad would silently corrupt training targets.
    """
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_values": f["input_values"]} for f in features]
        batch = self.processor.feature_extractor.pad(
            input_features, padding=self.padding, return_tensors="pt",
        )
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(
            label_features, padding=self.padding, return_tensors="pt",
        )
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        batch["labels"] = labels
        return batch


data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

# Sanity check
_test = data_collator([train_ds_prepared[0], train_ds_prepared[1]])
print("DataCollatorCTCWithPadding ready.")
print(f"  input_values shape : {tuple(_test['input_values'].shape)}")
print(f"  labels shape       : {tuple(_test['labels'].shape)}")
print(f"  labels has -100    : {(_test['labels'] == -100).any().item()}")

DataCollatorCTCWithPadding ready.
  input_values shape : (2, 16000)
  labels shape       : (2, 21)
  labels has -100    : True


In [8]:
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")


def compute_metrics(pred):
    pred_logits = pred.predictions
    label_ids   = pred.label_ids

    pred_ids = np.argmax(pred_logits, axis=-1)
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str  = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(label_ids, group_tokens=False)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)

    logger.info("--- Sample predictions (eval step) ---")
    for i in range(min(3, len(pred_str))):
        logger.info("  REF : %s", label_str[i])
        logger.info("  PRED: %s", pred_str[i])
        logger.info("  ---")
    logger.info("WER: %.4f  |  CER: %.4f", wer, cer)

    return {"wer": wer, "cer": cer}


print("WER + CER metrics loaded, compute_metrics ready.")

WER + CER metrics loaded, compute_metrics ready.


In [9]:
VOCAB_SIZE = processor.tokenizer.vocab_size

model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_NAME,
    # Regularisation
    attention_dropout=ATTENTION_DROPOUT,
    hidden_dropout=HIDDEN_DROPOUT,
    feat_proj_dropout=FEAT_PROJ_DROPOUT,
    # SpecAugment (EXP-1 values)
    mask_time_prob=MASK_TIME_PROB,
    mask_time_length=MASK_TIME_LENGTH,
    mask_feature_prob=MASK_FEAT_PROB,
    # CTC head
    ctc_loss_reduction="mean",
    ctc_zero_infinity=True,
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=VOCAB_SIZE,
    ignore_mismatched_sizes=True,
)

# CNN feature encoder: already well-trained on 436k multilingual hours, keep frozen
model.freeze_feature_encoder()

# Gradient checkpointing: ~20% slower, ~40% less VRAM — worth it below 24 GB
model.gradient_checkpointing_enable()

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model             : {MODEL_NAME}")
print(f"Total params      : {total:,}")
print(f"Trainable params  : {trainable:,}  ({100*trainable/total:.1f}%)")
print(f"Feature encoder   : FROZEN")
print(f"Gradient ckpt     : ENABLED")
print(f"mask_feature_prob : {model.config.mask_feature_prob}  (confirm = 0.1)")
print(f"mask_time_prob    : {model.config.mask_time_prob}")

2026-05-17 13:05:32 [INFO] httpx — HTTP Request: HEAD https://huggingface.co/facebook/wav2vec2-xls-r-300m/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-17 13:05:32 [INFO] httpx — HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/wav2vec2-xls-r-300m/1a640f32ac3e39899438a2931f9924c02f080a54/config.json "HTTP/1.1 200 OK"
2026-05-17 13:05:32 [INFO] httpx — HTTP Request: HEAD https://huggingface.co/facebook/wav2vec2-xls-r-300m/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
2026-05-17 13:05:32 [INFO] httpx — HTTP Request: HEAD https://huggingface.co/facebook/wav2vec2-xls-r-300m/resolve/main/model.safetensors.index.json "HTTP/1.1 404 Not Found"
2026-05-17 13:05:32 [INFO] httpx — HTTP Request: HEAD https://huggingface.co/facebook/wav2vec2-xls-r-300m/resolve/main/pytorch_model.bin "HTTP/1.1 302 Found"
2026-05-17 13:05:32 [INFO] httpx — HTTP Request: HEAD https://huggingface.co/facebook/wav2vec2-xls-r-300m/resolve/main/model.safetensors "HT

Loading weights: 100%|██████████| 422/422 [00:00<00:00, 13058.27it/s]

2026-05-17 13:05:33 [INFO] httpx — HTTP Request: GET https://huggingface.co/api/models/facebook/wav2vec2-xls-r-300m/discussions?p=0 "HTTP/1.1 200 OK"



[transformers] Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     | 
-----------------------------+------------+-
quantizer.weight_proj.weight | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
lm_head.weight               | MISSING    | 
lm_head.bias                 | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model             : facebook/wav2vec2-xls-r-300m
Total params      : 315,509,445
Trainable params  : 311,299,269  (98.7%)
Feature encoder   : FROZEN
Gradient ckpt     : ENABLED
mask_feature_prob : 0.1  (confirm = 0.1)
mask_time_prob    : 0.1


In [10]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # Schedule
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_steps=WARMUP,
    lr_scheduler_type="cosine",

    # Precision & speed
    fp16=FP16,
    dataloader_num_workers=4,

    # Eval & saving
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,

    # Logging
    logging_steps=100,
    report_to="none",

    # Misc
    seed=SEED,
    remove_unused_columns=False,
    label_names=["labels"],
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds_prepared,
    eval_dataset=eval_ds_prepared,
    processing_class=processor.feature_extractor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOP_PATIENCE,
            early_stopping_threshold=EARLY_STOP_THRESHOLD,
        )
    ],
)

logger.info("=" * 60)
logger.info("STARTING EXP-1 TRAINING")
logger.info("  Model             : %s", MODEL_NAME)
logger.info("  Dataset           : %s  (%d train samples)", DATASET_PATH, len(train_ds))
logger.info("  Epochs            : %d", EPOCHS)
logger.info("  LR                : %g", LR)
logger.info("  MASK_FEAT_PROB    : %g  (key fix vs baseline 0.004)", MASK_FEAT_PROB)
logger.info("  Effective batch   : %d x %d = %d", BATCH_SIZE, GRAD_ACCUM, BATCH_SIZE * GRAD_ACCUM)
logger.info("=" * 60)

try:
    trainer.train()
except KeyboardInterrupt:
    logger.warning("Training interrupted — saving checkpoint.")
    trainer.save_model(os.path.join(OUTPUT_DIR, "checkpoint-interrupted"))
    processor.save_pretrained(os.path.join(OUTPUT_DIR, "checkpoint-interrupted"))
    raise
except Exception as exc:
    logger.error("Training failed: %s", exc, exc_info=True)
    trainer.save_model(os.path.join(OUTPUT_DIR, "checkpoint-error"))
    processor.save_pretrained(os.path.join(OUTPUT_DIR, "checkpoint-error"))
    raise

# Save best model
FINAL_DIR = os.path.join(OUTPUT_DIR, "final")
trainer.save_model(FINAL_DIR)
processor.save_pretrained(FINAL_DIR)
logger.info("Final model saved to: %s", FINAL_DIR)

2026-05-17 13:05:33 [INFO] httpx — HTTP Request: GET https://huggingface.co/api/models/facebook/wav2vec2-xls-r-300m/commits/refs%2Fpr%2F36 "HTTP/1.1 200 OK"
2026-05-17 13:05:33 [INFO] httpx — HTTP Request: HEAD https://huggingface.co/facebook/wav2vec2-xls-r-300m/resolve/refs%2Fpr%2F36/model.safetensors.index.json "HTTP/1.1 404 Not Found"
2026-05-17 13:05:33 [INFO] httpx — HTTP Request: HEAD https://huggingface.co/facebook/wav2vec2-xls-r-300m/resolve/refs%2Fpr%2F36/model.safetensors "HTTP/1.1 302 Found"
2026-05-17 13:05:34 [INFO] telugu_asr_improved — ============================================================
2026-05-17 13:05:34 [INFO] telugu_asr_improved — STARTING EXP-1 TRAINING
2026-05-17 13:05:34 [INFO] telugu_asr_improved —   Model             : facebook/wav2vec2-xls-r-300m
2026-05-17 13:05:34 [INFO] telugu_asr_improved —   Dataset           : ./telugu_multi_dataset  (82755 train samples)
2026-05-17 13:05:34 [INFO] telugu_asr_improved —   Epochs            : 50
2026-05-17 13:05:3

Step,Training Loss,Validation Loss,Wer,Cer
500,13.951813,3.439350,1.000000,0.999994
1000,5.232625,0.823983,0.797697,0.204014
1500,3.621315,0.555638,0.658625,0.149054
2000,3.087211,0.463448,0.597110,0.129381
2500,2.930080,0.419427,0.571838,0.119799
3000,2.635685,0.386097,0.553307,0.113701
3500,2.506250,0.373377,0.525102,0.106618
4000,2.431697,0.352187,0.517121,0.103866
4500,2.344459,0.340153,0.503251,0.101082
5000,2.234734,0.324343,0.493474,0.097124


2026-05-17 13:16:42 [INFO] telugu_asr_improved — --- Sample predictions (eval step) ---
2026-05-17 13:16:42 [INFO] telugu_asr_improved —   REF : జ్ఞాన సముపార్షనలో పుస్తకాలు రచయితలు ప్రధాన పాత్ర పోషిస్తారని విజ్ఞానవంతులు రచయితలు సమాజానికి దిశానిర్దేశకులని వ్వాఖ్వానించారు
2026-05-17 13:16:42 [INFO] telugu_asr_improved —   PRED: ఁ
2026-05-17 13:16:42 [INFO] telugu_asr_improved —   ---
2026-05-17 13:16:42 [INFO] telugu_asr_improved —   REF : తొలిదశ మానవ ప్రయోగాల్లో భాగంగా సోమవారం ఇద్దరు వాలంటీర్లకు టీకా డోసు ఇచ్చిన సంగతి తెలిసిందే
2026-05-17 13:16:42 [INFO] telugu_asr_improved —   PRED: ఁ
2026-05-17 13:16:42 [INFO] telugu_asr_improved —   ---
2026-05-17 13:16:42 [INFO] telugu_asr_improved —   REF : సమావేశాసంతరం కేంద్ర ఆర్దిక మంతి అరుణ్జైట్లీ విలేకరులతో మాట్లాడుతూ
2026-05-17 13:16:42 [INFO] telugu_asr_improved —   PRED: ఁ
2026-05-17 13:16:42 [INFO] telugu_asr_improved —   ---
2026-05-17 13:16:42 [INFO] telugu_asr_improved — WER: 1.0000  |  CER: 1.0000


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.88s/it]


2026-05-17 13:28:05 [INFO] telugu_asr_improved — --- Sample predictions (eval step) ---
2026-05-17 13:28:05 [INFO] telugu_asr_improved —   REF : జ్ఞాన సముపార్షనలో పుస్తకాలు రచయితలు ప్రధాన పాత్ర పోషిస్తారని విజ్ఞానవంతులు రచయితలు సమాజానికి దిశానిర్దేశకులని వ్వాఖ్వానించారు
2026-05-17 13:28:05 [INFO] telugu_asr_improved —   PRED: ద్యానా సముపార్యంలో పుస్తకాలు రచేతులు ప్రధాన పాత్ర పోషిస్తారని విద్యనవంతులు రచేతులు సమాజానికి దిసానిర్దేసుకులని వ్యాక్యానించారుఁ
2026-05-17 13:28:05 [INFO] telugu_asr_improved —   ---
2026-05-17 13:28:05 [INFO] telugu_asr_improved —   REF : తొలిదశ మానవ ప్రయోగాల్లో భాగంగా సోమవారం ఇద్దరు వాలంటీర్లకు టీకా డోసు ఇచ్చిన సంగతి తెలిసిందే
2026-05-17 13:28:05 [INFO] telugu_asr_improved —   PRED: కులిదేశ మానవ ప్రయోగాల్లోభాగంగా సంవారం ఇత్తర్ వలంటేరులకు టీక డోసిచ్చిన సంగతి తెలుసిందేఁ
2026-05-17 13:28:05 [INFO] telugu_asr_improved —   ---
2026-05-17 13:28:05 [INFO] telugu_asr_improved —   REF : సమావేశాసంతరం కేంద్ర ఆర్దిక మంతి అరుణ్జైట్లీ విలేకరులతో మాట్లాడుతూ
2026-05-17 13:28:05

...

2026-05-17 18:26:52 [INFO] telugu_asr_improved — --- Sample predictions (eval step) ---
2026-05-17 18:26:52 [INFO] telugu_asr_improved —   REF : జ్ఞాన సముపార్షనలో పుస్తకాలు రచయితలు ప్రధాన పాత్ర పోషిస్తారని విజ్ఞానవంతులు రచయితలు సమాజానికి దిశానిర్దేశకులని వ్వాఖ్వానించారు
2026-05-17 18:26:52 [INFO] telugu_asr_improved —   PRED: జ్ఞాన సముపార్యంలో పుస్తకాలు రచయతులు ప్రధాన పాత్ర పోషిస్తారని విజ్ఞాన వంతులు రచయితలు సమాజానికి దిశానిర్దేశవులని వ్యాఖ్యానించారుఁ
2026-05-17 18:26:52 [INFO] telugu_asr_improved —   ---
2026-05-17 18:26:52 [INFO] telugu_asr_improved —   REF : తొలిదశ మానవ ప్రయోగాల్లో భాగంగా సోమవారం ఇద్దరు వాలంటీర్లకు టీకా డోసు ఇచ్చిన సంగతి తెలిసిందే
2026-05-17 18:26:52 [INFO] telugu_asr_improved —   PRED: తోల్లి దశ మానవ ప్రయోగాల్లో భాగంగా నోమవారం ఇద్దరు వాలంటీర్లకు టీకా డోస్ ఇచ్చిన సంగతి తెలిసిందేఁ
2026-05-17 18:26:52 [INFO] telugu_asr_improved —   ---
2026-05-17 18:26:52 [INFO] telugu_asr_improved —   REF : సమావేశాసంతరం కేంద్ర ఆర్దిక మంతి అరుణ్జైట్లీ విలేకరులతో మాట్లాడుతూ
2026-05-17 

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.75s/it]


2026-05-17 18:37:39 [INFO] telugu_asr_improved — --- Sample predictions (eval step) ---
2026-05-17 18:37:39 [INFO] telugu_asr_improved —   REF : జ్ఞాన సముపార్షనలో పుస్తకాలు రచయితలు ప్రధాన పాత్ర పోషిస్తారని విజ్ఞానవంతులు రచయితలు సమాజానికి దిశానిర్దేశకులని వ్వాఖ్వానించారు
2026-05-17 18:37:39 [INFO] telugu_asr_improved —   PRED: జ్ఞాన సముపాథ్యంలో పుస్తకాలు రచయితులు ప్రధాన పాత్ర పోషిస్తారని విజ్ఞానవంతులు రచయితలు సమాజానికి దిశానిర్గేశకులని వ్యాఖ్యానించారుఁ
2026-05-17 18:37:39 [INFO] telugu_asr_improved —   ---
2026-05-17 18:37:39 [INFO] telugu_asr_improved —   REF : తొలిదశ మానవ ప్రయోగాల్లో భాగంగా సోమవారం ఇద్దరు వాలంటీర్లకు టీకా డోసు ఇచ్చిన సంగతి తెలిసిందే
2026-05-17 18:37:39 [INFO] telugu_asr_improved —   PRED: తోలిదశ మానవ ప్రయోగాల్లో భాగంగా నోమవారం ఇద్తర వాలంటీర్లకు టీకా డోస్ ఇచ్చిన సంగతి తెలిసిందేఁ
2026-05-17 18:37:39 [INFO] telugu_asr_improved —   ---
2026-05-17 18:37:39 [INFO] telugu_asr_improved —   REF : సమావేశాసంతరం కేంద్ర ఆర్దిక మంతి అరుణ్జైట్లీ విలేకరులతో మాట్లాడుతూ
2026-05-17 18:3

Writing model shards: 100%|██████████| 1/1 [00:04<00:00,  4.17s/it]

2026-05-17 18:37:50 [INFO] telugu_asr_improved — Final model saved to: ./wav2vec2-telugu-improved/final


In [14]:
# ═══════════════════════════════════════════════════════════════════════════
# PART A — Greedy decoding evaluation (always runs)
# ═══════════════════════════════════════════════════════════════════════════

try:
    from transformers.utils.notebook import NotebookProgressCallback
    trainer.remove_callback(NotebookProgressCallback)
except Exception:
    pass

logger.info("Running greedy evaluation on eval set …")
final_metrics = trainer.evaluate()

final_wer = final_metrics.get("eval_wer", float("nan"))
final_cer = final_metrics.get("eval_cer", float("nan"))

print("=" * 60)
print("GREEDY DECODING — FINAL RESULTS")
print(f"  WER : {final_wer:.4f}  ({'✓' if final_wer < 0.40 else '✗'} target < 0.40)")
print(f"  CER : {final_cer:.4f}")
print(f"  Full metrics : {final_metrics}")
print("=" * 60)

metrics_path = os.path.join(OUTPUT_DIR, "final_eval_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(final_metrics, f, indent=2)
print(f"Metrics saved to : {metrics_path}")

# ═══════════════════════════════════════════════════════════════════════════
# PART B — KenLM 5-gram language model decoding
# ═══════════════════════════════════════════════════════════════════════════

USE_KENLM = True
KENLM_BIN = "./kenlm/build/bin"
ARPA_PATH = "./telugu_kenlm.arpa"
LM_TEXT   = "./telugu_lm.txt"

VOCAB_JSON = (
    "./vocab_v2.json"
    if Path("./vocab_v2.json").exists()
    else "./vocab.json"
)

if USE_KENLM:
    import pyctcdecode

    # 1. Dump all train transcripts for LM training
    transcripts = [ex["transcription"] for ex in train_ds]
    with open(LM_TEXT, "w", encoding="utf-8") as f:
        f.write("\n".join(transcripts))
    print(f"Wrote {len(transcripts)} transcripts to {LM_TEXT}")

    # 2. Train 5-gram ARPA with lmplz
    lmplz_bin = os.path.join(KENLM_BIN, "lmplz")
    assert Path(lmplz_bin).exists(), (
        f"lmplz not found at {lmplz_bin}.\n"
        "Build KenLM first: git clone https://github.com/kpu/kenlm && "
        "cd kenlm && mkdir build && cd build && cmake .. && make -j4"
    )
    print("Building 5-gram LM …")
    with open(LM_TEXT, "rb") as fin, open(ARPA_PATH, "w", encoding="utf-8") as fout:
        subprocess.run(
            [lmplz_bin, "-o", "5", "--discount_fallback"],
            stdin=fin, stdout=fout, check=True,
        )
    print(f"ARPA saved to : {ARPA_PATH}")

    # 3. Build pyctcdecode decoder
    with open(VOCAB_JSON) as f:
        vocab_dict = json.load(f)
    sorted_vocab = sorted(vocab_dict.keys(), key=lambda k: vocab_dict[k])

    decoder = pyctcdecode.build_ctcdecoder(
    sorted_vocab,
    ARPA_PATH,
    alpha=0.5,
    beta=1.0,
)
    print("KenLM decoder built.")

    # 4. Decode eval set with KenLM
    inference_model  = trainer.model.eval()
    inference_device = next(inference_model.parameters()).device
    kenlm_preds, kenlm_refs = [], []

    print(f"Decoding {len(eval_ds)} eval samples with KenLM …")
    for i, sample in enumerate(eval_ds):
        inputs = processor(
            sample["audio"]["array"],
            sampling_rate=16_000,
            return_tensors="pt",
            padding=True,
        )
        with torch.no_grad():
            logits = inference_model(
                input_values=inputs.input_values.to(inference_device)
            ).logits[0].cpu().numpy()
        kenlm_preds.append(decoder.decode(logits))
        kenlm_refs.append(sample["transcription"])
        if (i + 1) % 500 == 0:
            print(f"  {i+1}/{len(eval_ds)} decoded")

    kenlm_wer = wer_metric.compute(predictions=kenlm_preds, references=kenlm_refs)

    print()
    print("=" * 60)
    print("KENLM 5-GRAM DECODING RESULTS")
    print(f"  Greedy WER : {final_wer:.4f}")
    print(f"  KenLM WER  : {kenlm_wer:.4f}  (improvement: {(final_wer - kenlm_wer)*100:.1f}% abs.)")
    print("=" * 60)

    kenlm_metrics = {"greedy_wer": final_wer, "kenlm_wer": kenlm_wer}
    with open(os.path.join(OUTPUT_DIR, "kenlm_eval_metrics.json"), "w") as f:
        json.dump(kenlm_metrics, f, indent=2)
    print(f"KenLM metrics saved to: {os.path.join(OUTPUT_DIR, 'kenlm_eval_metrics.json')}")


2026-05-17 18:58:56 [INFO] telugu_asr_improved — Running greedy evaluation on eval set …
2026-05-17 19:00:51 [INFO] telugu_asr_improved — --- Sample predictions (eval step) ---
2026-05-17 19:00:51 [INFO] telugu_asr_improved —   REF : జ్ఞాన సముపార్షనలో పుస్తకాలు రచయితలు ప్రధాన పాత్ర పోషిస్తారని విజ్ఞానవంతులు రచయితలు సమాజానికి దిశానిర్దేశకులని వ్వాఖ్వానించారు
2026-05-17 19:00:51 [INFO] telugu_asr_improved —   PRED: జ్ఞాన సమూపాద్యంలో పుస్తకాలు రచయితులు ప్రధాన పాత్ర పోషిస్తారని విజ్ఞాన వంతులు రచయితలు సమాజానికి దిశానిర్గేశుకులని వ్యాఖ్యానించారుఁ
2026-05-17 19:00:51 [INFO] telugu_asr_improved —   ---
2026-05-17 19:00:51 [INFO] telugu_asr_improved —   REF : తొలిదశ మానవ ప్రయోగాల్లో భాగంగా సోమవారం ఇద్దరు వాలంటీర్లకు టీకా డోసు ఇచ్చిన సంగతి తెలిసిందే
2026-05-17 19:00:51 [INFO] telugu_asr_improved —   PRED: తోలి దశ మానవ ప్రయోగాల్లో భాగంగా సోమవారం ఇద్దరు వాలంటీర్లకు టీకా డోస్ ఇచ్చిన సంగతి తెలిసిందేఁ
2026-05-17 19:00:51 [INFO] telugu_asr_improved —   ---
2026-05-17 19:00:51 [INFO] telugu_asr_improve

=== 1/5 Counting and sorting n-grams ===
Reading /home/jovyan/work/asr_lrl_mscil_ftl/telugu_lm.txt
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************
Unigram tokens 784948 types 101177
=== 2/5 Calculating and sorting adjusted counts ===
Chain sizes: 1:1214124 2:10518364160 3:19721932800 4:31555090432 5:46017843200
Statistics:
1 101177 D1=0.714866 D2=1.03796 D3+=1.30858
2 469790 D1=0.85074 D2=1.19642 D3+=1.32527
3 634021 D1=0.927415 D2=1.33572 D3+=1.42749
4 634974 D1=0.960937 D2=1.49671 D3+=1.67904
5 582671 D1=0.921683 D2=1.69797 D3+=1.87752
Memory estimate for binary LM:
type    MB
probing 52 assuming -p 1.5
probing 62 assuming -r models -p 1.5
trie    26 without quantization
trie    15 assuming -q 8 -b 8 quantization 
trie    23 assuming -a 22 array pointer compression
trie    12 assuming -a 22 -q 8 -b 8 array pointer compression and qu

ARPA saved to : ./telugu_kenlm.arpa
2026-05-17 19:02:07 [INFO] pyctcdecode.decoder — Using arpa instead of binary LM file, decoder instantiation might be slow.
2026-05-17 19:02:07 [INFO] pyctcdecode.alphabet — Alphabet determined to be of regular style.
2026-05-17 19:02:07 [INFO] pyctcdecode.alphabet — Found '|' in vocabulary but not ' ', doing substitution.
2026-05-17 19:02:07 [INFO] pyctcdecode.alphabet — Found [PAD] in vocabulary, interpreted as a CTC blank token, substituting with .
2026-05-17 19:02:07 [INFO] pyctcdecode.alphabet — Found [UNK] in vocabulary, interpreting as unknown token, substituting with ⁇.
KenLM decoder built.
Decoding 9196 eval samples with KenLM …
  500/9196 decoded
  1000/9196 decoded
  1500/9196 decoded
  2000/9196 decoded
  2500/9196 decoded
  3000/9196 decoded
  3500/9196 decoded
  4000/9196 decoded
  4500/9196 decoded
  5000/9196 decoded
  5500/9196 decoded
  6000/9196 decoded
  6500/9196 decoded
  7000/9196 decoded
  7500/9196 decoded
  8000/9196 decoded